# 01 · Signal

*Part 1, section 2 and Part 2, section 1.* A trace says what happened. **Signal** says how it went.
It arrives from three places, and all of it lands on the trace as an **assessment** with a
source, so every later step reads and writes the same evidence:

| source | kind | here |
|---|---|---|
| HUMAN | feedback (`quality`: pass/fail + rationale), expectation (`expected_response`) | reviewers |
| CODE | deterministic checks | `examples/criteria.yaml` → `checks` |
| LLM_JUDGE | Guidelines judges, the `quality` judge | `criteria.yaml` → `guidelines`, `quality` |

In [ ]:
import _nb as nb

experiment_id = nb.setup()

In [ ]:
import mlflow
from selfimprove import feedback, signals, traces

week = traces.production()
print(len(week), "production traces")
t = week[0]
print("Q:", traces.question_of(t))
print("A:", traces.response_of(t)[:300])

**Under the hood:** `traces.production()` is one search and one filter: it drops every trace an
evaluation run created, so evaluation traffic never counts as production signal.

**MLflow in this step**

* `mlflow.search_traces(filter_string="tags.si_source = 'production'", order_by=["timestamp_ms DESC"],
  return_type="list")`: the production traces, newest first, as `Trace` objects
* trace metadata `mlflow.sourceRun`: MLflow sets it on traces that an evaluation run created; those are
  the ones dropped

## 1. Human feedback and expectations

Feedback says a trace is worth testing; an **expectation** says what correct means. A trace with
both is one call away from being a dataset record. Here is one label by hand, exactly as a review
UI or a thumbs-down button would log it:

In [ ]:
from mlflow.entities import AssessmentSource, AssessmentSourceType

me = AssessmentSource(source_type=AssessmentSourceType.HUMAN, source_id="you")
refund = next(t for t in week if "refund" in traces.question_of(t).lower())

mlflow.log_feedback(
    trace_id=refund.info.trace_id, name="quality", value=False, source=me,
    rationale="Quotes the old 30-day window; it has been 14 days since July 2026.",
)
mlflow.log_expectation(
    trace_id=refund.info.trace_id, name="expected_response", source=me,
    value="Refunds are available within 14 days of purchase.",
)
print(traces.human_feedback(mlflow.get_trace(refund.info.trace_id)))

Nobody labels a whole week by hand in a notebook. `examples/reviewer.py` plays three reviewers
(maria, dev, sam) who share an answer key but grade with slightly different rubrics, on purpose:
that is what real reviewer pools do. In real use this is `selfimprove label` in a terminal, or
`selfimprove review-queue` for labeling in the MLflow UI.

In [ ]:
from examples import reviewer

counts = reviewer.review_all(sample=0.75)   # label about 3/4 of the traffic
for name, (ok, bad) in counts.items():
    print(f"{name:<6} {ok:>3} pass {bad:>3} fail")

**Under the hood:** each reviewer verdict is a call to the harness's `feedback.label()`, which does
what the cell before did by hand. The request, response and spans stay as they were; the evidence
accumulates next to them.

**MLflow in this step**

* `mlflow.log_feedback(name="quality", value=True/False, rationale=...,
  source=AssessmentSource(source_type="HUMAN", source_id=<reviewer>))`: the verdict, stored on the
  trace as an assessment
* `mlflow.log_expectation(name="expected_response", ...)`: for a failure, the right answer, with the same
  source

In [ ]:
labeled = next(t for t in traces.production()
               if traces.human_feedback(t) is not None and t.info.trace_id != refund.info.trace_id)
for a in labeled.info.assessments:
    kind = "feedback" if a.feedback is not None else "expectation"
    print(f"{a.name:<18} {kind:<12} {str(a.value)[:45]:<46} {a.source.source_type}:{a.source.source_id}")

## 2. Automated signal: checks and judges

From cheapest and most trustworthy to most flexible. The criteria file defines all of them; this is
what it builds:

In [ ]:
print(open("examples/criteria.yaml").read()[:2200])

In [ ]:
for s in signals.live():
    print(f"{type(s).__name__:<22} {s.name}")
print("must-pass:", signals.must_pass())

**Under the hood:** `signals.live()` only builds scorer objects; it calls no model and writes
nothing. The judges need credentials, so in offline mode the list holds only the code scorers.

**MLflow in this step**

* `@scorer`: each `checks` entry becomes a decorated function. `grounded_in_policy` is one too: it
  inspects TOOL spans and then asks a `make_judge()` judge
* `Guidelines(name=..., guidelines=...)`: one judge per `guidelines` entry
* `make_judge(name="quality", instructions=..., feedback_value_type=bool, model=SI_JUDGE_MODEL)`: the
  `quality` judge

A code check is a plain function over the response (or the whole trace, for tool checks):

In [ ]:
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer

@scorer
def no_stale_refund_window(outputs) -> Feedback:
    stale = "30 day" in str(outputs).lower() or "30-day" in str(outputs).lower()
    return Feedback(value=not stale, rationale="Mentions the old 30-day window." if stale else "OK")

no_stale_refund_window(outputs="You can get a refund within 30 days of purchase.")

### A cheap first pass: Jev (Part 3, section 3)

Reviewer labels arrive unevenly, and a text judge on every trace gets expensive. A **decision
model** such as TypeSafe's Jev (MLflow 3.17+) returns a yes/no with a **probability** instead of
generated text, cheap enough to judge most traffic. It gives no rationale, so verdicts in an
uncertain probability band go to the text judge, which supplies one. With
`SI_FAST_JUDGE_MODEL=typesafe:/jev-latest` and `TYPESAFE_API_KEY` in `.env`, the harness's
`quality` judge and the `grounded_in_policy` check both work this way.

In [ ]:
from mlflow.genai.judges import make_judge
from selfimprove import config

if config.settings.has_fast_judge:
    jev = make_judge(
        name="quality",
        instructions="Answer yes or no. Would a careful support lead send the answer in {{ outputs }} "
                     "to the customer who asked {{ inputs }} as-is?",
        feedback_value_type=bool,
        model="typesafe:/jev-latest",
    )
    t = week[0]
    verdict = jev(inputs={"question": traces.question_of(t)}, outputs=traces.response_of(t))
    print("value:", verdict.value, "| p(pass):", signals.probability(verdict), "| rationale:", verdict.rationale)
    print("re-judged by the text judge when p(pass) is in", config.settings.fast_judge_band)
else:
    print("No first-pass model configured (SI_FAST_JUDGE_MODEL + TYPESAFE_API_KEY); the text judge does it all.")

The grounded check goes one step further: it inspects the `lookup_policy` TOOL span first. A
missing, failed, or empty lookup fails the trace for free; only real lookups reach a model, which
is asked whether the answer stays within the policy text the tool returned.

In [ ]:
import inspect
print(inspect.getsource(signals.make_grounded).split('"""', 2)[2][:1400])

## 3. Judge the week's traces in place

`evaluate()` with **no `predict_fn`** scores the existing traces: nothing is re-run, and the verdicts
land on the traces as assessments (source CODE or LLM_JUDGE) next to the reviewers'.
(`selfimprove score` does this for new traces since its last run.)

In [ ]:
result = mlflow.genai.evaluate(data=week, scorers=signals.live())
week = traces.production()   # re-read: the traces now carry the verdicts

**Under the hood:** the harness's `score` job makes exactly this call on traces that have no verdict
yet.

**MLflow in this step**

* `mlflow.genai.evaluate(data=<traces>, scorers=...)` with no `predict_fn`: reads each trace's inputs,
  outputs and spans, runs every scorer, and logs each verdict back onto that same trace as a feedback
  assessment
* assessment source types: `@scorer` functions get `CODE`, judges get `LLM_JUDGE`
* the evaluation run: the call also creates an MLflow run that holds one aggregate metric per scorer
  (`<name>/mean`)

In [ ]:
print("evaluation run:", result.run_id)
print({k: round(float(v), 2) for k, v in result.metrics.items()})
print()
for a in week[0].info.assessments:
    print(f"{a.name:<22} {str(a.value):<6} {a.source.source_type}:{a.source.source_id}")

In [ ]:
nb.verdicts(week, names=["quality", "no_stale_policy", "no_disclaimer_opener", "grounded_in_policy",
                          "leads_with_answer", "no_guessing"]).head(12)

In [ ]:
# With a first pass configured: how many quality verdicts did the text judge have to make?
rejudged = [a for t in week for a in t.info.assessments
            if a.name == "quality" and (a.metadata or {}).get("si.escalated") == "true"]
firsts = [a for t in week for a in t.info.assessments if a.name == "quality" and "si.escalated" in (a.metadata or {})]
if firsts:
    print(f"{len(rejudged)} of {len(firsts)} quality verdicts were re-judged by {config.settings.judge_model}")

In [ ]:
must = signals.must_pass()
judged = [t for t in week if traces.has_verdict(t, must)]
print(f"failure rate on v1: {sum(traces.failed(t, must) for t in judged) / len(judged):.0%} of {len(judged)} traces")

## 4. Signal is searchable

Assessments are queryable like any other trace field. Boolean feedback is stored as text, so filter
with `'false'`, not `False`.

In [ ]:
failing = mlflow.search_traces(
    filter_string="tags.si_source = 'production' AND feedback.quality = 'false' "
                  "AND expectation.expected_response IS NOT NULL",
    return_type="list",
)
print(len(failing), "traces failed by someone, with the right answer attached")

One caveat before trusting any of this: the `quality` judge writes under the **same name** as
the reviewers. That is deliberate (it lets the harness measure how often they agree, in
notebook 05), but it means a search for `feedback.quality = 'false'` matches judge verdicts too.
The harness's helpers (`traces.human_feedback`, `traces.failed`) always check the source.

**Next:** [02 · Store](02_store.ipynb), where the failures become a test suite.